# MEMC cvičení 1 – Úvod do Monte Carla

---

V tomto sešitu uvidíte celou metodu Monte Carlo na třech nejjednodušších úlohách:

- odhad čísla $\pi$ (trefa/netrefa)
- odhad $\pi$ z Buffonovy jehly
- odhad integrálu $\int_0^1 e^{-x^2}\,dx$

a hlavně si odnesete jedno pravidlo, které platí v celém kurzu:

> **Chyba Monte Carlo odhadu klesá jako $1/\sqrt{N}$.** Chcete desetkrát přesnější výsledek? Potřebujete stokrát víc bodů.

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def estimate(sampler, n):
    '''Běžné Monte Carlo: průměr n volání generátoru a jeho směrodatná chyba sigma/sqrt(n).'''
    s = s2 = 0.0
    for _ in range(n):
        x = sampler()
        s += x
        s2 += x * x
    mean = s / n
    var = max(s2 / n - mean * mean, 0.0)
    return mean, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    '''Vypíše odhad a ověří, že leží do k směrodatných chyb od přesné hodnoty.'''
    print(f"{name}: {mean:.5f} +- {se:.5f}  (přesně {exact:.5f})")
    assert abs(mean - exact) <= k * se, name

Funkce `estimate` bere **generátor** (funkci bez argumentů, která při každém volání vrátí jeden náhodný vzorek) a počet vzorků $n$. Vrací průměr a směrodatnou chybu $\sigma/\sqrt{n}$, kde $\sigma$ odhadneme ze samotných vzorků. Funkce `check` použijeme k ověření výsledků proti přesné hodnotě.

## Hlavní myšlenka

Chceme spočítat nějakou veličinu $\theta$. Najdeme náhodnou veličinu $X$ takovou, že $E X = \theta$, a pak

$$\theta = E X \approx \bar X_N = \frac{1}{N}\sum_{i=1}^N X_i, \qquad \text{chyba} \approx \frac{\sigma}{\sqrt{N}}.$$

Podle CLV leží přesná hodnota s pravděpodobností $\approx 95\,\%$ v intervalu spolehlivosti $\bar X_N \pm 2\,\sigma/\sqrt{N}$. Celé umění Monte Carla je v tom vymyslet vhodné $X$.

## 1. Odhad $\pi$ metodou trefa/netrefa

Bod $(X, Y)$ rozdělený rovnoměrně v jednotkovém čtverci $[0,1]^2$ padne do čtvrtkruhu $x^2 + y^2 \le 1$ s pravděpodobností rovnou poměru obsahů:

$$P(X^2 + Y^2 \le 1) = \frac{\pi/4}{1} = \frac{\pi}{4}.$$

Zvolíme tedy $X_i = 4\cdot\mathbf 1[\text{bod padl do čtvrtkruhu}]$. Pak $E X_i = \pi$ a $X_i$ nabývá jen hodnot $0$ a $4$.

In [ ]:
def pi_hit():
    x = random.random()
    y = random.random()
    return 4.0 if x * x + y * y <= 1 else 0.0


print(f"{'N':>9s} {'odhad pi':>10s} {'|chyba|':>10s} {'sm. chyba':>10s}  chyba v 2 sm. chybách?")
for e in range(1, 7):
    n = 10 ** e
    m, se = estimate(pi_hit, n)
    print(f"{n:9,d} {m:10.5f} {abs(m - math.pi):10.5f} {se:10.5f}  {'ano' if abs(m - math.pi) <= 2 * se else 'ne'}".replace(",", " "))

m, se = estimate(pi_hit, 1_000_000)
check("pi", m, se, math.pi)

Než se podíváme na čísla, **obrázek toho, co děláme**: vlevo body v jednotkovém čtverci, zeleně ty, které padly do čtvrtkruhu. Podíl zelených je $\pi/4$. Vpravo běžící odhad $\pi$ z jednoho dlouhého běhu: s rostoucím $N$ se usazuje a pás $\pm 2\,\sigma/\sqrt{N}$ (interval spolehlivosti) se zužuje.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# vlevo: 2000 bodů ve čtverci
M_ = 2000
uvnitr, venku = [], []
for _ in range(M_):
    x, y = random.random(), random.random()
    (uvnitr if x * x + y * y <= 1 else venku).append((x, y))
ax = axes[0]
ax.scatter(*zip(*venku), s=5, c="tab:red", alpha=0.6, label="netrefa")
ax.scatter(*zip(*uvnitr), s=5, c="tab:green", alpha=0.6, label="trefa")
ax.plot([math.cos(t / 100 * math.pi / 2) for t in range(101)], [math.sin(t / 100 * math.pi / 2) for t in range(101)], "k")
ax.set_aspect("equal")
ax.set_title(f"{M_} bodů: trefa {len(uvnitr) / M_:.1%} (přesně $\\pi/4$ = {math.pi / 4:.1%}), odhad $\\pi$ = {4 * len(uvnitr) / M_:.3f}")
ax.legend(loc="lower left")

# vpravo: běžící odhad s pásem +- 2 sm. chyby
N_ = 100_000
s = 0
beh, pas = [], []
for i in range(1, N_ + 1):
    x, y = random.random(), random.random()
    s += x * x + y * y <= 1
    p = s / i
    beh.append(4 * p)
    pas.append(4 * 2 * math.sqrt(max(p * (1 - p), 1e-12) / i))
idx = list(range(1, N_ + 1))
axes[1].semilogx(idx, beh, label="běžící odhad $\\hat\\pi_N$")
axes[1].fill_between(idx, [b - w for b, w in zip(beh, pas)], [b + w for b, w in zip(beh, pas)], alpha=0.25, label="$\\pm 2\\,\\sigma/\\sqrt{N}$")
axes[1].axhline(math.pi, color="k", ls=":", label="$\\pi$")
axes[1].set_ylim(2.6, 3.7)
axes[1].set_xlabel("počet bodů $N$")
axes[1].set_title("Odhad se usazuje, pás se zužuje")
axes[1].legend()
plt.tight_layout()
plt.show()

Chyba klesá s $N$ pomalu. Ukažme to na grafu: pro každé $N$ zopakujeme odhad 20× a průměrnou absolutní chybu vyneseme v logaritmických osách. Přímka se sklonem $-1/2$ znamená chybu $\propto 1/\sqrt{N}$.

In [ ]:
sigma_pi = 4 * math.sqrt(math.pi / 4 * (1 - math.pi / 4))  # přesná sigma veličiny X_i

Ns = [10 ** e for e in range(1, 6)]
opakovani = 20
prum_chyba = []
for n in Ns:
    chyby = [abs(estimate(pi_hit, n)[0] - math.pi) for _ in range(opakovani)]
    prum_chyba.append(sum(chyby) / opakovani)

plt.figure(figsize=(6.5, 4.5))
plt.loglog(Ns, prum_chyba, "o-", label="průměrná |chyba| ze 20 běhů")
plt.loglog(Ns, [sigma_pi / math.sqrt(n) for n in Ns], "r--", label=r"$\sigma/\sqrt{N}$")
plt.xlabel("počet bodů N")
plt.ylabel("chyba")
plt.title(r"Chyba odhadu $\pi$")
plt.legend()
plt.show()

print(f"sigma = {sigma_pi:.4f}; pro chybu 0.001 (2 sm. chyby) potřebujeme N = {(2 * sigma_pi / 0.001) ** 2:,.0f}".replace(",", " "))

Naměřená chyba leží rovnoběžně s přímkou $\sigma/\sqrt{N}$, jen o něco pod ní: průměrná *absolutní* chyba je zhruba $0.8\,\sigma/\sqrt{N}$. Důležitý je sklon.

Tři platné cifry $\pi$ stojí zhruba **deset milionů bodů**. Na odhad $\pi$ je tedy Monte Carlo opravdu špatná volba. Síla metody je jinde: chyba $\sim 1/\sqrt{N}$ **nezávisí na dimenzi** úlohy (viz bod 5).

## 2. Buffonova jehla

Na podlaze jsou rovnoběžné čáry ve vzdálenosti $D$. Házíme jehlu délky $L \le D$. Označme:

- $x \in [0, D/2]$ vzdálenost středu jehly od nejbližší čáry,
- $\theta \in [0, \pi/2]$ úhel mezi jehlou a čárami.

Jehla čáru protne, když $x \le \tfrac{L}{2}\sin\theta$. Dvojice $(x, \theta)$ je rovnoměrná na obdélníku $[0, D/2]\times[0, \pi/2]$, takže

$$p = P(\text{protne}) = \frac{1}{\frac D2\cdot\frac\pi2}\int_0^{\pi/2}\frac L2\sin\theta\,d\theta = \frac{2L}{\pi D}
\quad\Longrightarrow\quad \pi = \frac{2L}{D\,p}.$$

Odhadneme $p$ jako podíl zásahů a dosadíme. Směrodatnou chybu $\pi$ dostaneme ze směrodatné chyby $p$: relativní chyba se přenese, $\operatorname{se}(\hat\pi) \approx \hat\pi\cdot\operatorname{se}(\hat p)/\hat p$ (tzv. delta metoda).

Jak to vypadá: vlevo jehly na podlaze (červeně ty, které čáru protnou), vpravo **prostor možností** $(\theta, x)$ s oblastí $x \le \tfrac L2\sin\theta$. Pravděpodobnost protnutí je podíl plochy pod křivkou k ploše celého obdélníku, přesně totéž jako u čtvrtkruhu.

In [ ]:
L_, D_ = 1.0, 2.0
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

# vlevo: 25 jehel mezi čárami
ax = axes[0]
for k in range(-1, 3):
    ax.axhline(k * D_, color="k", lw=1)
for _ in range(25):
    cx, cy = random.random() * 8, random.random() * 2 * D_
    th = random.random() * math.pi
    dist = min(cy % D_, D_ - cy % D_)
    protne = dist <= L_ / 2 * abs(math.sin(th))
    dx, dy = L_ / 2 * math.cos(th), L_ / 2 * math.sin(th)
    ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy], color="tab:red" if protne else "tab:blue", lw=2)
ax.set_aspect("equal")
ax.set_ylim(-0.5, 2 * D_ + 0.5)
ax.set_title("Jehly délky $L = 1$, čáry po $D = 2$ (červeně: jehla protíná čáru)")

# vpravo: prostor (theta, x) a oblast zásahu
ax = axes[1]
ths = [i / 200 * math.pi / 2 for i in range(201)]
ax.fill_between(ths, [L_ / 2 * math.sin(t) for t in ths], color="tab:red", alpha=0.35, label="protne: $x \\leq \\frac{L}{2}\\sin\\theta$")
ax.plot(ths, [L_ / 2 * math.sin(t) for t in ths], "k")
ax.plot([0, math.pi / 2, math.pi / 2, 0, 0], [0, 0, D_ / 2, D_ / 2, 0], "b--", label="všechny možnosti $[0,\\pi/2]\\times[0,D/2]$")
ax.set_xlabel(r"úhel $\theta$")
ax.set_ylabel("vzdálenost středu od čáry $x$")
ax.set_title("Pravděpodobnost = podíl ploch")
ax.set_ylim(0, 1.3)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def buffon_hit(L=1.0, D=2.0):
    x = random.random() * D / 2             # vzdálenost středu jehly od nejbližší čáry
    theta = random.random() * math.pi / 2   # úhel jehly (pozor: používáme pi!)
    return 1.0 if x <= L * math.sin(theta) / 2 else 0.0


def buffon_pi(n, L=1.0, D=2.0):
    p, se = estimate(lambda: buffon_hit(L, D), n)
    pi_hat = 2 * L / (D * p)
    return pi_hat, pi_hat * se / p


print(f"{'N':>9s} {'odhad pi':>10s} {'|chyba|':>10s} {'sm. chyba':>10s}")
for e in range(2, 7):
    n = 10 ** e
    m, se = buffon_pi(n)
    print(f"{n:9,d} {m:10.5f} {abs(m - math.pi):10.5f} {se:10.5f}".replace(",", " "))

Při $L = 1,\ D = 2$ je $p = 1/\pi \approx 0.318$ a konvergence je ještě pomalejší než u předchozí metody.

> **Čestná poznámka:** náš simulátor úhel $\theta$ generuje pomocí `math.pi`, takže „počítat $\pi$ pomocí $\pi$" je vlastně podvod. Skutečný experiment by úhel nepotřeboval. Jde nám tu o cvičení z odvození pravděpodobnosti a o ukázku, že stejnou konstantu lze získat z jiné úlohy.

## 3. Integrál jako střední hodnota

Je-li $\gamma \sim R(a, b)$ rovnoměrné na $[a, b]$, pak $E f(\gamma) = \frac{1}{b-a}\int_a^b f(x)\,dx$. Proto

$$I = \int_a^b f(x)\,dx = (b - a)\,E f(\gamma) \approx (b-a)\,\frac{1}{N}\sum_{i=1}^N f(\gamma_i),
\qquad \operatorname{se} = (b-a)\,\frac{\sigma_f}{\sqrt{N}}.$$

Vyzkoušíme to na $f(x) = e^{-x^2}$ na $[0, 1]$. Přesnou hodnotu známe ze vztahu s chybovou funkcí: $I = \tfrac{\sqrt\pi}{2}\operatorname{erf}(1)$.

In [ ]:
def mc_integral(f, a=0.0, b=1.0, n=10_000):
    '''Odhad integrálu f na [a, b] jako (b - a) * průměr f(γ), γ ~ R(a, b). Vrací (odhad, sm. chyba).'''
    m, se = estimate(lambda: f(a + (b - a) * random.random()), n)
    return (b - a) * m, (b - a) * se


f = lambda x: math.exp(-x * x)
presne = math.sqrt(math.pi) / 2 * math.erf(1)

print(f"přesně I = {presne:.6f}")
print(f"{'N':>9s} {'odhad':>10s} {'|chyba|':>10s} {'sm. chyba':>10s}")
for e in range(1, 7):
    n = 10 ** e
    m, se = mc_integral(f, 0.0, 1.0, n)
    print(f"{n:9,d} {m:10.6f} {abs(m - presne):10.6f} {se:10.6f}".replace(",", " "))

m, se = mc_integral(f, 0.0, 1.0, 1_000_000)
check("integral", m, se, presne)

### Dva způsoby, jak odhadnout totéž

Integrál lze odhadnout i **trefou/netrefou** jako u $\pi$: bodem $(x, y)$ v obdélníku $[a,b]\times[0,C]$ a podílem těch, které padnou pod graf, vynásobeným obsahem obdélníku $C(b-a)$. Který způsob je přesnější při stejném $N$?

Obrázek obou způsobů vedle sebe: **vlevo** průměr funkčních hodnot (v $N$ náhodných bodech změříme výšku grafu a průměrná výška krát šířka je odhad plochy), **vpravo** trefa/netrefa (náhodné body v obdélníku, podíl pod grafem krát plocha).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
grid = [i / 200 for i in range(201)]
ax = axes[0]
us = [random.random() for _ in range(12)]
vyska = sum(f(u) for u in us) / len(us)
ax.fill_between(grid, [vyska] * len(grid), alpha=0.15, color="tab:orange", label=f"obdélník (b - a) · průměr = {vyska:.3f}")
ax.plot(grid, [f(x) for x in grid], "k", lw=2, label="$f(x) = e^{-x^2}$")
ax.vlines(us, 0, [f(u) for u in us], color="tab:blue", alpha=0.7, label="$f(U_i)$ v 12 náhodných bodech")
ax.axhline(vyska, color="tab:orange", ls="--")
ax.set_title("Průměr funkčních hodnot")
ax.legend(loc="lower left", fontsize=8)

ax = axes[1]
pod, nad = [], []
for _ in range(600):
    x, y = random.random(), random.random()
    (pod if y < f(x) else nad).append((x, y))
ax.scatter(*zip(*nad), s=6, c="tab:red", alpha=0.6, label="nad grafem")
ax.scatter(*zip(*pod), s=6, c="tab:green", alpha=0.6, label="pod grafem")
ax.plot(grid, [f(x) for x in grid], "k", lw=2)
ax.set_title(f"Trefa/netrefa: podíl pod grafem {len(pod) / 600:.0%} · plocha 1")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def integral_trefa(f, C, a=0.0, b=1.0, n=10_000):
    '''Trefa/netrefa: podíl bodů pod grafem f v obdélníku [a, b] x [0, C] krát jeho obsah.'''
    plocha = C * (b - a)
    m, se = estimate(lambda: 1.0 if C * random.random() < f(a + (b - a) * random.random()) else 0.0, n)
    return plocha * m, plocha * se


n = 100_000
m1, se1 = mc_integral(f, 0.0, 1.0, n)
m2, se2 = integral_trefa(f, 1.0, 0.0, 1.0, n)
print(f"průměr f(γ):       {m1:.6f} +- {se1:.6f}")
print(f"trefa/netrefa:     {m2:.6f} +- {se2:.6f}")
print(f"poměr chyb = {se2 / se1:.2f}, tedy trefa/netrefa potřebuje asi {(se2 / se1) ** 2:.1f}x víc bodů")

Průměr funkčních hodnot má menší rozptyl, protože každý bod přispívá **hodnotou** $f(\gamma_i)$, zatímco trefa/netrefa z něj udělá jen ano/ne. Tuto myšlenku (snižování rozptylu) rozvineme ve cvičení 7.

## 4. Kolik bodů potřebuji?

Chceme chybu nejvýše $\delta$ se spolehlivostí 95 %. Z $k_\alpha\,\sigma/\sqrt{N} \le \delta$ dostaneme

$$N \ge \left(\frac{k_\alpha\,\sigma}{\delta}\right)^2, \qquad k_\alpha \approx 1.96.$$

Potíž: $\sigma$ neznáme. Řešení je **pilotní běh**: malý počet vzorků, z něj odhadneme $\sigma \approx \operatorname{se}\cdot\sqrt{n}$, pak dopočítáme potřebné $N$.

In [ ]:
n_pilot = 10_000
_, se = mc_integral(f, 0.0, 1.0, n_pilot)
sigma = se * math.sqrt(n_pilot)
print(f"odhad sigma z pilotního běhu: {sigma:.4f}\n")

for delta in (1e-2, 1e-3, 1e-4):
    N = math.ceil((1.96 * sigma / delta) ** 2)
    print(f"delta = {delta:g}: N >= {N:,d}".replace(",", " "))

# ověření pro delta = 1e-3
delta = 1e-3
N = math.ceil((1.96 * sigma / delta) ** 2)
m, _ = mc_integral(f, 0.0, 1.0, N)
print(f"\nskutečná chyba pro N = {N:,d}: {abs(m - presne):.6f}  (cíl {delta})".replace(",", " "))

Každé zpřesnění o řád ($\delta \to \delta/10$) znamená **100× víc bodů**. Kdo potřebuje přesnost $10^{-8}$, měl by sáhnout po jiné metodě (u jednorozměrného integrálu třeba po lichoběžníkovém pravidle). Monte Carlo se vyplatí, až když je dimenze velká.

## 5. Dimenze: objem koule

Objem jednotkové koule v $d$ rozměrech známe: $V_d = \dfrac{\pi^{d/2}}{\Gamma(d/2 + 1)}$. Odhadneme ho trefou/netrefou v krychli $[-1, 1]^d$ (objem $2^d$): $X = 2^d\cdot\mathbf 1[\,\lVert x\rVert \le 1\,]$.

In [ ]:
def objem_koule(d):
    '''Generátor veličiny X = 2^d * 1[bod v jednotkové kouli], bod rovnoměrný v [-1, 1]^d.'''
    def sampler():
        r2 = 0.0
        for _ in range(d):
            x = 2 * random.random() - 1
            r2 += x * x
        return 2.0 ** d if r2 <= 1 else 0.0
    return sampler


n = 100_000
print(f"{'d':>2s} {'odhad':>10s} {'přesně':>10s} {'sm. chyba':>10s} {'rel. chyba':>11s}")
for d in range(2, 7):
    m, se = estimate(objem_koule(d), n)
    presne_d = math.pi ** (d / 2) / math.gamma(d / 2 + 1)
    assert abs(m - presne_d) <= 5 * se
    print(f"{d:2d} {m:10.4f} {presne_d:10.4f} {se:10.4f} {se / presne_d:10.2%}")

Stejné $N = 100\,000$ dává v šesti rozměrech použitelný výsledek. Mřížkové metody (kvadratury) by při $k$ bodech na osu potřebovaly $k^d$ bodů, což s dimenzí roste exponenciálně. U Monte Carla se s dimenzí zhoršuje jen konstanta $\sigma$ (zde: zásahy jsou čím dál vzácnější, relativní chyba pomalu roste), nikoli exponent $-1/2$.

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Monte Carlo odhad | $\theta = E X \approx \bar X_N$ |
| Směrodatná chyba | $\sigma/\sqrt{N}$, $\sigma$ odhadneme ze vzorků (funkce `estimate`) |
| Interval spolehlivosti | $\bar X_N \pm 2\,\sigma/\sqrt{N}$ (z CLV, spolehlivost $\approx 95\,\%$) |
| Integrál | $\int_a^b f \approx (b-a)\cdot\overline{f(\gamma)}$ |
| Trefa/netrefa vs. průměr | průměr $f(\gamma)$ má menší rozptyl než trefa/netrefa |
| Kolik bodů | $N \ge (k_\alpha\sigma/\delta)^2$; 10× přesněji = 100× víc bodů |
| Dimenze | chyba $\propto 1/\sqrt{N}$ nezávisí na dimenzi, proto se Monte Carlo používá pro vícerozměrné úlohy |

### Zkuste si

1. Odhadněte $\int_0^2 \sin(x)\,dx$ (přesně $1 - \cos 2 \approx 1.416$) funkcí `mc_integral` a ověřte ji pomocí `check`.
2. V Buffonově úloze změňte $L$ a $D$ (třeba $L = D = 1$). Jak se změní $p$ a jak přesný je odhad $\pi$ při stejném $N$?
3. Změňte `d` v objemu koule na $10$. Kolik zásahů z $100\,000$ dostanete a jaká je relativní chyba? Co to říká o hodnotě trefa/netrefa ve vysoké dimenzi?